[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S03_funciones_python_moderno.ipynb)

# Sesión 03 · Funciones y Python moderno

**Módulo 1: Python** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Escribir funciones con `def` y `return`, con parámetros posicionales, por nombre y por defecto.
2. Hacer funciones que resistan casos borde: listas vacías, negativos, ceros y búsquedas sin coincidencias.
3. Importar módulos y usar funciones de `math`.
4. Escribir comprensiones de listas, expresiones condicionales y `lambda`, y usarlas con `sorted`, `map` y `filter`.
5. Controlar errores con `try` y `except`.

## 📋 Qué debes saber antes
Lo de las sesiones 1 y 2: tipos, textos, listas, tuplas, diccionarios, `if` y bucles.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Hoy los verificadores **llaman a tus funciones** con varios casos, incluidos casos borde; si alguno falla, verás con qué entrada.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
import re
import statistics

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
ventas_semana = [int(v) for v in rng.integers(800, 5000, size=7)]
meta_diaria = 3000

_NOMBRES = ["polo", "jean", "casaca", "gorra", "medias", "chompa", "bufanda"]
_stock = [int(s) for s in rng.integers(1, 30, size=7)]
_stock[2] = 0
_stock[5] = 0
productos = [(n, round(float(rng.uniform(15, 150)), 2), s) for n, s in zip(_NOMBRES, _stock)]

# ---------- Datos de práctica: movimientos bancarios ----------
lineas_extracto = []
for _d in range(1, 16):
    _f = f"2026-09-{_d:02d}"
    if _d == 1:
        lineas_extracto.append(f"{_f};sueldo;{rng.uniform(2500, 3500):.2f}")
    elif _d == 4:
        lineas_extracto.append(f"{_f};cajero;N/A")
    elif _d == 7:
        lineas_extracto.append(f"{_f};servicios")
    elif _d == 10:
        lineas_extracto.append("")
    elif _d == 12:
        lineas_extracto.append(f"  {_f} ; Restaurante ; -{rng.uniform(20, 300):.2f}  ")
    elif _d == 13:
        lineas_extracto.append(f"{_f};transferencia;{rng.uniform(100, 600):.2f}".replace(".", ","))
    else:
        _c = str(rng.choice(["supermercado", "restaurante", "cajero", "servicios"]))
        _m = f"-{rng.uniform(20, 300):.2f}"
        if _d % 3 == 0:
            _m = _m.replace(".", ",")
        lineas_extracto.append(f"{_f};{_c};{_m}")

_D = copy.deepcopy({k: globals()[k] for k in ["dias", "ventas_semana", "meta_diaria", "productos", "lineas_extracto"]})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _num(texto):
    """Referencia por expresiones regulares (no por float + try)."""
    t = texto.strip()
    m = re.fullmatch(r"(-?)(\d+)(?:[.,](\d+))?", t)
    if not m:
        return None
    signo = -1 if m.group(1) else 1
    decimales = m.group(3) or ""
    return signo * (int(m.group(2)) + (int(decimales) / 10 ** len(decimales) if decimales else 0))


def _linea(linea):
    m = re.fullmatch(r"\s*([^;]*?)\s*;\s*([^;]*?)\s*;\s*([^;]*?)\s*", linea)
    if not m or _num(m.group(3)) is None:
        return None
    return (m.group(1), m.group(2).upper(), _num(m.group(3)))


def _movs(lineas):
    salida = []
    for l in lineas:
        t = _linea(l)
        if t is not None:
            salida.append(t)
    return salida


def _gastos(movs):
    g = {}
    for _, c, x in movs:
        if x < 0:
            g[c] = g.get(c, 0) + abs(x)
    return {c: round(x, 2) for c, x in g.items()}


def _top(movs, n=3):
    egresos = [(m[2], i, m) for i, m in enumerate(movs) if m[2] < 0]
    egresos.sort()
    return [m for _, _, m in egresos[:max(n, 0)]]


def check_ejercicio_1():
    r = _Revision("Ejercicio 1")
    f = r.funcion("precio_con_igv")
    if f is not _FALTA:
        ref = lambda p, igv=0.18: None if p < 0 else round(p + p * igv, 2)
        casos = [((100,), {}), ((59.9,), {}), ((0,), {}), ((80,), {"igv": 0.10}), ((50, 0), {}), ((-5,), {})]
        for args, kw in casos:
            texto = "precio_con_igv(" + ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()]) + ")"
            motivo = "con precio negativo debía devolver None" if args[0] < 0 else "no es lo esperado"
            r.caso(texto, f, args, kw, ref(*args, **kw), motivo=motivo)
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    f = r.funcion("resumen_ventas")
    v = _D["ventas_semana"]

    def ref(xs):
        if len(xs) == 0:
            return (0, 0.0, None)
        return (math.fsum(xs), round(math.fsum(xs) / len(xs), 2), sorted(xs)[-1])

    if f is not _FALTA:
        for xs in (v, [100], [], [-50, 20, 30], [0, 0]):
            motivo = "con lista vacía debía devolver (0, 0.0, None)" if not xs else "no es lo esperado (se espera una tupla de 3 valores)"
            r.caso(f"resumen_ventas({xs!r})", f, (xs,), esperado=ref(xs), motivo=motivo)
    t, p, m = ref(v)
    r.valor("total", t, None, "desempaqueta el resultado de llamar a tu función con `ventas_semana`", igual=lambda a, b: _igual(a, b, 0.0051))
    r.valor("promedio", p, None, "desempaqueta el resultado de llamar a tu función con `ventas_semana`", igual=lambda a, b: _igual(a, b, 0.0051))
    r.valor("maximo", m, None, "desempaqueta el resultado de llamar a tu función con `ventas_semana`", igual=lambda a, b: _igual(a, b, 0.0051))
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_resultado": "d024f6472cd1381eeddb28a59daaff16528770c069c615713e06c20d6b4498fd",
        "pred_contador": "0e939e9ca064726168df510a5b86d75df221609e471cb3d8c10019b5aff835d1",
        "pred_n": "345e42ad061bc48d0359a369b1c6c567a1c82108b53e556c40e79578d7443105",
        "pred_retorno": "dd4b095932d31cacdde75692eefa4d23c54556ab1c4bb7c317df662973a2d0d9",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    f = r.funcion("variacion_pct")
    if f is not _FALTA:
        for a, b in ((100, 120), (200, 150), (80, 80), (0, 50), (-100, -50), (4050, 1174)):
            esperado = None if a <= 0 else round((b - a) * 100 / a, 2)
            motivo = "con `anterior` en 0 o negativo debía devolver None" if a <= 0 else "no es lo esperado"
            r.caso(f"variacion_pct({a}, {b})", f, (a, b), esperado=esperado, motivo=motivo)
    g = r.funcion("primer_indice_sobre")
    if g is not _FALTA:
        v = _D["ventas_semana"]
        casos = [(v, 3000), (v, 1000), (v, 10**6), ([], 5), ([5, 1, 9], 9), ([-3, -1], -2)]
        for xs, u in casos:
            esperado = next((i for i in range(len(xs)) if xs[i] >= u), -1)
            motivo = "cuando no hay coincidencias debía devolver -1" if esperado == -1 else "no es lo esperado"
            r.caso(f"primer_indice_sobre({_corto(xs, 30)}, {u})", g, (xs, u), esperado=esperado, motivo=motivo)
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    f = r.funcion("cajas_necesarias")
    if f is not _FALTA:
        casos = [((25,), {}), ((24,), {}), ((1,), {}), ((0,), {}), ((-3,), {}), ((10,), {"por_caja": 4})]
        for args, kw in casos:
            u, pc = args[0], kw.get("por_caja", 12)
            esperado = 0 if u <= 0 else -(-u // pc)
            texto = "cajas_necesarias(" + ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()]) + ")"
            r.caso(texto, f, args, kw, esperado, motivo="con 0 o menos unidades debía devolver 0" if u <= 0 else "no es lo esperado")
    g = r.funcion("desviacion")
    if g is not _FALTA:
        for xs in (_D["ventas_semana"], [2, 4, 4, 4, 5, 5, 7, 9], [10], [], [-5, 5]):
            esperado = None if not xs else round(statistics.pstdev(xs), 2)
            r.caso(f"desviacion({_corto(xs, 40)})", g, (xs,), esperado=esperado,
                   motivo="con lista vacía debía devolver None" if not xs else "no es lo esperado")
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    v, meta = _D["ventas_semana"], _D["meta_diaria"]
    con_igv = []
    for x in v:
        con_igv.append(round(x + x * 0.18, 2))
    r.valor("con_igv", con_igv, list, "cada venta debería estar multiplicada por 1.18 y redondeada a 2 decimales",
            igual=lambda a, b: _igual(a, b, 0.0051))
    r.valor("sobre_meta", [x for x in v if not x < meta], list,
            "deberían ser las ventas que alcanzan `meta_diaria`, en el orden original")
    r.valor("etiquetas", [("baja", "alta")[x >= meta] for x in v], list, "revisa la expresión condicional y el límite (`>=`)")
    r.valor("dias_sobre_meta", [d for d, x in zip(_D["dias"], v) if not x < meta], list,
            "deberían ser los nombres de los días que alcanzan la meta")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_vacia": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_ninguno": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
    })
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    ps = _D["productos"]
    por_precio = [p for _, p in sorted((p[1], p) for p in ps)]
    decorados = sorted((-(p[1] * p[2]), i, p) for i, p in enumerate(ps))
    r.valor("por_precio", por_precio, list, "deberían estar ordenados por precio, de menor a mayor")
    r.valor("por_valor_desc", [p for _, _, p in decorados], list,
            "deberían estar ordenados por precio × stock, de mayor a menor")
    r.valor("nombres_mayus", [p[0].upper() for p in ps], list, "deberían ser los nombres en MAYÚSCULAS, en el orden original")
    r.valor("agotados", [p for p in ps if not p[2]], list, "deberían ser las tuplas completas de los productos con stock 0")
    r.fin()


def check_ejercicio_7():
    r = _Revision("Ejercicio 7")
    f = r.funcion("a_numero")
    if f is not _FALTA:
        for t in ("12.5", " 7 ", "12,5", "-30.10", "", "abc", "N/A", "3,1,4"):
            esperado = _num(t)
            motivo = "si el texto no es un número debía devolver None" if esperado is None else "no es lo esperado"
            r.caso(f"a_numero({t!r})", f, (t,), esperado=esperado, motivo=motivo)
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    lineas = _D["lineas_extracto"]
    ref_movs = _movs(lineas)
    f = r.funcion("parsear_linea")
    if f is not _FALTA:
        extra = ["a;b", "2026-09-30;pago;abc", "2026-09-30;Pago;  -10,5 ", "x;y;z;1"]
        for l in lineas + extra:
            esperado = _linea(l)
            motivo = "para una línea inválida debía devolver None" if esperado is None else "no es lo esperado"
            r.caso(f"parsear_linea({l!r})", f, (l,), esperado=esperado, motivo=motivo)
    g = r.funcion("limpiar_extracto")
    if g is not _FALTA:
        r.caso("limpiar_extracto(lineas_extracto)", g, (lineas,), esperado=ref_movs)
        r.caso("limpiar_extracto([])", g, ([],), esperado=[])
        r.caso('limpiar_extracto(["", "x;y"])', g, (["", "x;y"],), esperado=[])
    r.valor("movs", ref_movs, list, "debería ser el resultado de `limpiar_extracto(lineas_extracto)`",
            igual=lambda a, b: _igual(a, b, 0.0051))
    h = r.funcion("gasto_por_concepto")
    if h is not _FALTA:
        solo_ingresos = [m for m in ref_movs if m[2] > 0]
        r.caso("gasto_por_concepto(movs)", h, (ref_movs,), esperado=_gastos(ref_movs))
        r.caso("gasto_por_concepto([])", h, ([],), esperado={})
        r.caso("gasto_por_concepto(<solo ingresos>)", h, (solo_ingresos,), esperado={})
    t = r.funcion("top_egresos")
    if t is not _FALTA:
        for args, kw in (((ref_movs,), {}), ((ref_movs,), {"n": 1}), ((ref_movs,), {"n": 50}), (([],), {}), ((ref_movs,), {"n": 0})):
            texto = "top_egresos(" + ("movs" if args[0] else "[]") + "".join(f", {k}={v}" for k, v in kw.items()) + ")"
            r.caso(texto, t, args, kw, _top(*args, **kw))
    _sin = globals().get("lineas_extracto") == _D["lineas_extracto"]
    if not _sin:
        r.mal("`lineas_extracto` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    f = r.funcion("normalizar")
    if f is not _FALTA:
        for xs in (_D["ventas_semana"], [10, 20, 30], [], [5, 5, 5], [-2, 0, 2]):
            if not xs:
                esperado = []
            elif len(set(xs)) == 1:
                esperado = [0.0] * len(xs)
            else:
                lo, hi = sorted(xs)[0], sorted(xs)[-1]
                esperado = [round((x - lo) / (hi - lo), 4) for x in xs]
            r.caso(f"normalizar({_corto(xs, 40)})", f, (xs,), esperado=esperado, tol=0.00006)
    g = r.funcion("ranking")
    if g is not _FALTA:
        d = dict(zip(_D["dias"], _D["ventas_semana"]))
        orden = [k for _, k in sorted((-x, k) for k, x in d.items())]
        r.caso("ranking(ventas_por_dia)", g, (d,), esperado=orden[:3])
        r.caso("ranking(ventas_por_dia, n=2, ascendente=True)", g, (d,), {"n": 2, "ascendente": True},
               esperado=[k for _, k in sorted((x, k) for k, x in d.items())][:2])
        r.caso("ranking({})", g, ({},), esperado=[])
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("dias          =", dias)
print("ventas_semana =", ventas_semana)
print("meta_diaria   =", meta_diaria)
print("productos (nombre, precio, stock):")
for p in productos:
    print("  ", p)
print()
print("🏦 Líneas del extracto bancario (tal como llegaron)")
for linea in lineas_extracto:
    print("  ", repr(linea))

---
## 1. Funciones: `def`, `return` y parámetros

### 📘 Concepto
Una **función** empaqueta código con nombre para reutilizarlo:

```python
def nombre(parametro_1, parametro_2=valor_por_defecto):
    ...
    return resultado
```

- `return` **devuelve** un valor a quien llamó a la función y la termina. `print` solo muestra en pantalla: no deja nada para seguir calculando.
- Al llamar, los argumentos pueden ir **por posición** (`f(3, 10)`) o **por nombre** (`f(precio=10, unidades=3)`). Los que van por nombre pueden ir en cualquier orden, pero siempre después de los posicionales.
- Un parámetro **por defecto** (`descuento=0`) se puede omitir al llamar a la función.

In [ ]:
def importe(unidades, precio, descuento=0):
    bruto = unidades * precio
    return bruto - bruto * descuento / 100

print(importe(3, 10))                  # por posición, descuento por defecto
print(importe(3, 10, 20))              # los tres por posición
print(importe(precio=10, unidades=3, descuento=50))   # por nombre

resultado_ej = importe(2, 25) + importe(1, 5)   # el valor devuelto se puede seguir usando
print(resultado_ej)

### ✍️ Tu turno · Ejercicio 1: precio con IGV
Define `precio_con_igv(precio, igv=0.18)`:
- Devuelve el precio con el impuesto aplicado, redondeado a 2 decimales.
- `igv` es la tasa como decimal (0.18 significa 18 %) y por defecto vale 0.18.
- Caso borde: si `precio` es negativo, devuelve `None`.

El verificador la probará, entre otros casos, con `precio_con_igv(80, igv=0.10)` y con un precio de 0.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Pon el caso borde al principio: un `if` que, si el precio es negativo, haga `return None`. Así el resto de la función solo trabaja con precios válidos.
</details>

<details><summary>💡 Pista 2</summary>

Aplicar una tasa `t` es multiplicar por `(1 + t)`. Usa `return`, no `print`, y redondea con `round(..., 2)`.
</details>

---
## 2. Varios valores de retorno y alcance de las variables

### 📘 Concepto
- `return a, b, c` devuelve una **tupla**. Quien llama puede desempaquetarla: `x, y, z = f(...)`.
- Las variables creadas **dentro** de una función son **locales**: solo existen mientras la función se ejecuta y no cambian las variables de fuera, aunque se llamen igual.
- Una función sin `return` devuelve `None`.
- Reasignar un parámetro dentro de la función no cambia la variable original que se pasó.

In [ ]:
def min_max(valores):
    return min(valores), max(valores)

bajo_ej, alto_ej = min_max([7, 2, 9])
print(bajo_ej, alto_ej)

tasa = 0.05
def calcular_interes(monto):
    tasa = 0.10          # variable local: no toca la tasa de fuera
    return monto * tasa

print(calcular_interes(100), tasa)

### ✍️ Tu turno · Ejercicio 2: resumen y alcance
**Parte A.** Define `resumen_ventas(ventas)`, que devuelva la tupla `(total, promedio, maximo)`:
- `promedio` redondeado a 2 decimales.
- Caso borde: si la lista está vacía, devuelve `(0, 0.0, None)`.

Luego llámala con `ventas_semana` y desempaqueta el resultado en `total`, `promedio` y `maximo`.

**Parte B.** Predice **sin ejecutar**:
```python
contador = 10
def sumar_uno():
    contador = 0
    contador += 1
    return contador
resultado = sumar_uno()

def duplicar(x):
    x = x * 2

n = 5
retorno = duplicar(n)
```

| Variable | Pregunta |
|---|---|
| `pred_resultado` | ¿cuánto vale `resultado`? |
| `pred_contador` | ¿cuánto vale `contador` al final? |
| `pred_n` | ¿cuánto vale `n` al final? |
| `pred_retorno` | ¿cuánto vale `retorno`? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Con una lista vacía, `sum` funciona pero dividir entre `len` y `max` fallan. Revisa ese caso antes de calcular nada.
</details>

<details><summary>💡 Pista 2</summary>

Empieza con `if len(ventas) == 0:` (o `if not ventas:`) y devuelve la tupla del caso borde. En la parte B fíjate en qué variables son locales y en si `duplicar` tiene `return`.
</details>

---
## 3. Funciones que resisten casos borde

### 📘 Concepto
Los datos reales traen sorpresas: listas vacías, ceros, negativos, búsquedas que no encuentran nada. Una función robusta los resuelve **al principio**, con `return` tempranos (se llaman **cláusulas de guarda**), y deja el caso normal para el final.

Decide y documenta qué devolver en cada caso: `None` cuando no hay respuesta posible, `0` cuando la cuenta es cero, `-1` cuando una búsqueda no encuentra nada.

In [ ]:
def ticket_promedio(total, clientes):
    if clientes <= 0:        # guarda: evita dividir entre 0
        return None
    return round(total / clientes, 2)

def buscar_producto(nombres, buscado):
    for i, nombre in enumerate(nombres):
        if nombre == buscado:
            return i         # return dentro del bucle: termina en cuanto encuentra
    return -1                # solo llega aquí si no lo encontró

print(ticket_promedio(900, 12), ticket_promedio(900, 0))
print(buscar_producto(["polo", "jean"], "jean"), buscar_producto(["polo", "jean"], "gorra"), buscar_producto([], "polo"))

### ✍️ Tu turno · Ejercicio 3: variación y búsqueda
1. `variacion_pct(anterior, actual)`: variación porcentual de `anterior` a `actual`, redondeada a 2 decimales (de 100 a 120 es `20.0`; de 200 a 150 es `-25.0`). Si `anterior` es 0 o negativo, devuelve `None`.
2. `primer_indice_sobre(ventas, umbral)`: la posición de la primera venta mayor o igual que `umbral`. Si no hay ninguna (o la lista está vacía), devuelve `-1`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

La variación porcentual es la diferencia (actual menos anterior) dividida entre el anterior, por 100.
</details>

<details><summary>💡 Pista 2</summary>

En `primer_indice_sobre`, usa `enumerate` y haz `return i` en cuanto encuentres una venta que cumpla. El `return -1` va **después** del bucle, no dentro de un `else`.
</details>

---
## 4. Módulos: `import` y `math`

### 📘 Concepto
Un **módulo** es un archivo con funciones listas para usar. Hay tres formas de importarlo:

| Forma | Cómo se usa después |
|---|---|
| `import math` | `math.sqrt(16)` |
| `import math as m` | `m.sqrt(16)` (alias: así se usa `import numpy as np`) |
| `from math import sqrt, ceil` | `sqrt(16)`, `ceil(2.1)` |

Del módulo `math`: `sqrt` (raíz cuadrada), `ceil` (redondea hacia arriba), `floor` (hacia abajo), `pi`, `log`, `isclose`.

In [ ]:
import math
from math import ceil

print(math.sqrt(81), math.floor(2.9), ceil(2.1), round(math.pi, 4))
print(ceil(10 / 3), 10 // 3)     # hacia arriba vs división entera

### ✍️ Tu turno · Ejercicio 4: cajas y dispersión
1. `cajas_necesarias(unidades, por_caja=12)`: cuántas cajas hacen falta para empacar **todas** las unidades (una caja a medio llenar cuenta). Si `unidades` es 0 o negativo, devuelve `0`. Usa `math`.
2. `desviacion(valores)`: desviación estándar poblacional redondeada a 2 decimales, es decir, la raíz cuadrada del promedio de `(x - media) ** 2`. Si la lista está vacía, devuelve `None`. Usa `math.sqrt`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Para las cajas, la división normal da decimales: ¿qué función de `math` los lleva al entero de arriba?
</details>

<details><summary>💡 Pista 2</summary>

Para la desviación: calcula la media, luego acumula `(x - media) ** 2` para cada valor, divide entre la cantidad de valores (no entre la cantidad menos uno) y saca la raíz.
</details>

---
## 5. Comprensiones de listas y expresión condicional

### 📘 Concepto
Una **comprensión de lista** crea una lista en una línea:

```python
[expresion for x in coleccion]              # transforma cada elemento
[expresion for x in coleccion if condicion] # y además filtra
```

Una **expresión condicional** elige entre dos valores en una línea: `a if condicion else b`. Se puede usar dentro de una comprensión: `["par" if x % 2 == 0 else "impar" for x in numeros]`.

Ojo: el `if` al final **filtra** (quita elementos); el `if ... else` al principio **elige** un valor para cada elemento.

In [ ]:
unidades_ej = [4, 0, 7, 3, 0]
print([u * 10 for u in unidades_ej])                          # transforma
print([u for u in unidades_ej if u > 0])                      # filtra
print(["agotado" if u == 0 else "ok" for u in unidades_ej])   # elige
print([p.upper() for p, u in zip(["a", "b", "c"], [1, 0, 2]) if u > 0])

### ✍️ Tu turno · Ejercicio 5: comprensiones sobre la semana
**Parte A.** Crea cada lista con **una** comprensión:
1. `con_igv`: cada venta de `ventas_semana` por 1.18, redondeada a 2 decimales.
2. `sobre_meta`: solo las ventas que alcanzan `meta_diaria`.
3. `etiquetas`: `"alta"` si la venta alcanza `meta_diaria` y `"baja"` si no.
4. `dias_sobre_meta`: los nombres de los días (de `dias`) cuya venta alcanza la meta.

**Parte B · casos borde.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_vacia` | ¿qué valor da `len([x * 2 for x in []])`? |
| `pred_ninguno` | ¿qué valor da `len([x for x in [1, 2, 3] if x > 5])`? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

`sobre_meta` usa un `if` al final (filtra). `etiquetas` usa `if ... else` al principio (elige).
</details>

<details><summary>💡 Pista 2</summary>

Para `dias_sobre_meta` recorre dos listas a la vez con `zip` dentro de la comprensión, y quédate con el día.
</details>

---
## 6. `lambda`, `sorted(key=...)`, `map` y `filter`

### 📘 Concepto
- `lambda x: expresion` es una función corta, sin nombre, de una sola expresión. Se usa sobre todo como argumento de otra función.
- `sorted(datos, key=f)` ordena según lo que devuelve `f` para cada elemento. Con `reverse=True`, de mayor a menor. Si dos elementos empatan, conservan su orden original.
- `map(f, datos)` aplica `f` a cada elemento; `filter(f, datos)` se queda con los elementos en los que `f` da `True`. Los dos devuelven un objeto perezoso: envuélvelos en `list(...)` para ver el resultado.

In [ ]:
vendedores_ej = [("Ana", 1200), ("Luis", 800), ("Rosa", 1500)]

print(sorted(vendedores_ej, key=lambda v: v[1]))                # por monto
print(sorted(vendedores_ej, key=lambda v: v[0], reverse=True))  # por nombre, de Z a A
print(list(map(lambda v: v[1] * 2, vendedores_ej)))
print(list(filter(lambda v: v[1] >= 1000, vendedores_ej)))

### ✍️ Tu turno · Ejercicio 6: ordenar y filtrar productos
`productos` es una lista de tuplas `(nombre, precio, stock)`. Crea:
1. `por_precio`: los productos ordenados por precio, de menor a mayor.
2. `por_valor_desc`: ordenados por valor del inventario (precio × stock), de mayor a menor.
3. `nombres_mayus`: los nombres en MAYÚSCULAS, usando `map`.
4. `agotados`: las tuplas de los productos con stock 0, usando `filter`.

Usa `lambda` en los cuatro.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Dentro de la `lambda`, `p` es una tupla completa: `p[0]` es el nombre, `p[1]` el precio y `p[2]` el stock.
</details>

<details><summary>💡 Pista 2</summary>

Para `por_valor_desc`, la `key` devuelve `p[1] * p[2]` y además hace falta `reverse=True`. `map` y `filter` necesitan `list(...)` alrededor.
</details>

---
## 7. Controlar errores: `try` y `except`

### 📘 Concepto
Si un código puede fallar (por ejemplo, convertir a número un texto sucio), en lugar de dejar que el programa se detenga puedes **atrapar** el error:

```python
try:
    ...          # lo que puede fallar
except ValueError:
    ...          # qué hacer si falla con ese tipo de error
```

Atrapa solo el error que esperas (`ValueError`, `ZeroDivisionError`, `KeyError`...). Un `except` sin tipo esconde también los errores de tu propio código.

In [ ]:
def a_entero(texto):
    try:
        return int(texto)
    except ValueError:
        return None

print(a_entero("42"), a_entero("cuarenta"), a_entero(""))

try:
    resultado_ej = 10 / 0
except ZeroDivisionError:
    resultado_ej = None
print(resultado_ej)

### ✍️ Tu turno · Ejercicio 7: convertir texto a número
Define `a_numero(texto)`:
- Quita los espacios de los lados y acepta tanto punto como coma decimal: `"12,5"` y `" 12.5 "` dan `12.5`.
- Devuelve un `float`.
- Si el texto no se puede convertir (vacío, `"abc"`, `"N/A"`...), devuelve `None`. Usa `try`/`except`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_7()

<details><summary>💡 Pista 1</summary>

Primero prepara el texto (quitar espacios, cambiar la coma por punto) y después intenta convertirlo dentro del `try`.
</details>

<details><summary>💡 Pista 2</summary>

¿Qué error lanza `float("abc")`? Ese es el que debes atrapar. Mira también qué pasa con `"3,1,4"` después de cambiar las comas por puntos.
</details>

---
## 🏋️ Reto final: limpiar un extracto bancario
`lineas_extracto` trae las líneas de un extracto tal como llegaron: el formato es `fecha;concepto;monto`, pero hay líneas vacías, líneas con campos faltantes, montos que no son números, espacios de sobra y coma decimal.

Define estas funciones (puedes reutilizar `a_numero`):
1. `parsear_linea(linea)`: devuelve la tupla `(fecha, CONCEPTO, monto)` con los textos sin espacios a los lados, el concepto en MAYÚSCULAS y el monto como `float`. Si la línea no tiene exactamente 3 campos o el monto no es un número, devuelve `None`.
2. `limpiar_extracto(lineas)`: devuelve la lista de tuplas válidas, en el orden original. Luego crea `movs = limpiar_extracto(lineas_extracto)`.
3. `gasto_por_concepto(movs)`: diccionario `concepto → total gastado`, solo con egresos (montos negativos), en positivo y redondeado a 2 decimales. Sin egresos, `{}`.
4. `top_egresos(movs, n=3)`: lista con los `n` movimientos de monto más negativo, del más negativo al menos. Si hay menos egresos que `n`, devuelve todos; si `n` es 0 o negativo, `[]`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`parsear_linea` es un `split(";")`, una revisión de la cantidad de partes y una conversión con `a_numero`. `limpiar_extracto` cabe en una comprensión con filtro.
</details>

<details><summary>💡 Pista 2</summary>

Para `top_egresos`: filtra los egresos, ordénalos con `sorted(..., key=lambda m: m[2])` y toma los primeros `n` con slicing. Revisa qué hace el slicing `[:0]` y con un `n` negativo.
</details>

---
## 🚀 Nivel pro (opcional)
1. `normalizar(valores)`: lleva cada valor a la escala 0–1 con `(x - mínimo) / (máximo - mínimo)`, redondeado a 4 decimales. Lista vacía → `[]`. Si todos los valores son iguales, devuelve una lista de `0.0` del mismo largo.
2. `ranking(diccionario, n=3, ascendente=False)`: devuelve las `n` claves con mayor valor (o menor, si `ascendente=True`), ordenadas. Pruébala con `dict(zip(dias, ventas_semana))`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar la diferencia entre `return` y `print` dentro de una función.
- [ ] Llamar a una función con argumentos por posición, por nombre y usando valores por defecto.
- [ ] Devolver varios valores y desempaquetarlos.
- [ ] Explicar qué es una variable local y qué devuelve una función sin `return`.
- [ ] Escribir cláusulas de guarda para listas vacías, ceros, negativos y búsquedas sin resultado.
- [ ] Importar un módulo de tres formas distintas.
- [ ] Explicar la diferencia entre el `if` que filtra y el `if ... else` que elige dentro de una comprensión.
- [ ] Ordenar una lista de tuplas con `sorted(key=lambda ...)`.
- [ ] Atrapar un `ValueError` con `try`/`except`.

**Próxima sesión (S04):** empezamos NumPy con el `ndarray`.